# Heuristic Solution

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Sets and Parameters Inicialized

In [2]:
df = pd.read_csv('Nodos_26.csv',header=None)

# The distances are converted to km
df = df / 1000

In [3]:
nodes = [1,2,5,6,7,8,11,12,13,14,15,16,17,'base',19,20,22,23,24,25,26,27,28,29,30,31]
df.columns = nodes
df.index = nodes

In [4]:
d = [0.4,0.52,0.56,0.19,0.28,0.6,0.67,0.47,0.97,0.34,0.98,0.64,0.11,0,0.92,0.38,0.82,0.53,0.64,0.05,0.75,0.28,0.64,0.54,0.76,0.4]
d = pd.DataFrame(d, index=nodes, columns=['Demanda'])
d

,Demanda
1,0.40
2,0.52
5,0.56
6,0.19
7,0.28
8,0.60
11,0.67
12,0.47
13,0.97
14,0.34


In [5]:
# The velocity is fixed to 20
v = 20

## Proposed Algorithm

In [6]:
demands = d.copy()
n0 = 'base'
ruta = [n0]
rutas = []
costos = []
while demands.sum()["Demanda"] > 0:
    carga = 1
    ruta = [n0]
    costo = 1
    while carga > 0 and demands["Demanda"].sum() > 0:
        n = df.loc[n0, np.array(demands["Demanda"]>0)].idxmax()
        ruta.append(n)
        
        if carga > demands.loc[n].Demanda:
            carga -= demands.loc[n].Demanda
            demands.loc[n].Demanda = 0
            costo += df.loc[ruta[-2],n]/v
            
        elif carga == demands.loc[n].Demanda:
            carga = 0
            demands.loc[n].Demanda = 0
            ruta.append(n0)
            costo += df.loc[ruta[-2],n0]/v
            
        else:
            demands.loc[n].Demanda -= carga
            carga = 0
            ruta.append(n0)
            costo += df.loc[ruta[-2],n0]/v
            
        while carga > 0 and demands["Demanda"].sum() > 0:
            if df.loc[df.loc[n, np.array(demands["Demanda"]>0)].idxmin(),n] < df.loc[n,'base']:
                n = df.loc[n, np.array(demands["Demanda"]>0)].idxmin()

                ruta.append(n)
                
                if carga > demands.loc[n].Demanda:
                    carga -= demands.loc[n].Demanda
                    demands.loc[n].Demanda = 0
                    costo += df.loc[ruta[-2],n]/v
                    
                elif carga == demands.loc[n].Demanda:
                    carga = 0
                    demands.loc[n].Demanda = 0
                    ruta.append(n0)
                    costo += df.loc[ruta[-2],n0]/v
                    
                else:
                    demands.loc[n].Demanda -= carga
                    carga = 0
                    ruta.append(n0)
                    costo += df.loc[ruta[-2],n0]/v
                    
            else:
                carga = 0
                ruta.append(n0)
                costo += df.loc[ruta[-2],n0]/v
            
        
    rutas.append(ruta)
    costos.append(costo)
if rutas[-1][-1] != n0:
    rutas[-1].append(n0)
    costo += df.loc[ruta[-2],n0]
    
    
rutas

[['base', 1, 5, 11, 'base'],
 ['base', 6, 2, 8, 'base'],
 ['base', 7, 8, 14, 15, 'base'],
 ['base', 26, 25, 19, 'base'],
 ['base', 27, 28, 29, 'base'],
 ['base', 20, 22, 'base'],
 ['base', 13, 22, 'base'],
 ['base', 29, 30, 'base'],
 ['base', 11, 12, 'base'],
 ['base', 19, 16, 'base'],
 ['base', 22, 23, 24, 'base'],
 ['base', 12, 31, 15, 'base'],
 ['base', 30, 24, 'base'],
 ['base', 16, 17, 'base'],
 ['base', 15, 'base']]

## Calculating Costs

### Decimal part

In [7]:
decimal_part_hours = 0
for i in range(len(costos)):
    decimal_part_hours += costos[i]
print("The cost of the decimal past is:" + str(decimal_part_hours))

The cost of the decimal past is:16.10944814618


### Integer part

Nodes 3,4,9,10,21

In [8]:
whole_distances = pd.read_csv('distances.csv',header=0,index_col=0)
whole_distances = whole_distances / 1000

integer_demands = pd.DataFrame([5,7,8,8,7,4,6,7,8,8,7,1,7,7,5,5,6,0,4,1,8,7,5,5,5,4,1,6,6,6,5],index=whole_distances.index,columns=['Demanda'])

integer_cost_hours = whole_distances['18'] * integer_demands['Demanda'] / v * 2 + integer_demands['Demanda']
print("The cost of the integer part is:", str(integer_cost_hours.sum()))

The cost of the integer part is: 182.6795204385851


### Total Cost

In [9]:
print("The total cost is: " ,str((integer_cost_hours.sum()) + (decimal_part_hours)))

The total cost is:  198.7889685847651


In [10]:
total_cost_day = (integer_cost_hours.sum()/8) + (decimal_part_hours/8)
print("The minimun days in which the problem can be solved is: " , np.ceil(total_cost_day))

The minimun days in which the problem can be solved is:  25.0


## Associate Each Route with the Corresponding Cost
So it can be processed with the model of ordering routes

In [11]:
df_route_costs_decimal = pd.DataFrame({'Route': [route[1:] for route in rutas]})

#Add the cost of each route to the dataframe
df_route_costs_decimal['Cost'] = costos

df_route_costs_integer = pd.DataFrame({'Route': whole_distances.index, 'Cost': integer_cost_hours})
df_route_costs = pd.concat([df_route_costs_integer,df_route_costs_decimal], ignore_index=True)
df_route_costs

,Route,Cost
0,1,5.700583
1,2,7.890674
2,3,8.959426
3,4,8.923429
4,5,7.790000
5,6,4.519466
6,7,6.667692
7,8,7.652993
8,9,8.654928
9,10,8.593406


## Optimizing Routes Order

In [12]:
# Create a new list to store the modified rows
new_rows = []

# Iterate over the dataframe
for index, row in df_route_costs.iterrows():
    cost = row['Cost']
    route = row['Route']
    
    # Check if the integer part of the cost is even and greater than 0
    if int(cost) % 2 == 0 and int(cost) != 0:
        integer_part = int(cost)
        decimal_part = cost - integer_part
        
        # Divide the cost into equal parts based on the integer part
        divided_cost = 1 + (decimal_part / integer_part)
        
        # Append the route multiple times based on the integer part
        for _ in range(integer_part):
            new_rows.append({'Route': route, 'Cost': divided_cost})
    else:
        # If the cost is not even, retain the original row
        new_rows.append({'Route': route, 'Cost': cost})

# Convert new_rows to a DataFrame
final_df = pd.DataFrame(new_rows)

# Display the resulting dataframe
final_df

,Route,Cost
0,1,5.700583
1,2,7.890674
2,3,1.119928
3,3,1.119928
4,3,1.119928
...,...,...
110,"[22, 23, 24, base]",1.052408
111,"[12, 31, 15, base]",1.060397
112,"[30, 24, base]",1.051137
113,"[16, 17, base]",1.047466


### Export to CSVs

In [13]:
# Save the resulting dataframe to a CSV file
df_route_costs.to_csv('Costos_rutas.csv',index=False)
final_df.to_csv('rutas_costos.csv', index=False)